# Day 5 seed spread on Kaggle

Attach the private `octlm-code` dataset and the completed `octlm-day4` notebook output. Run with one T4. The notebook trains three seeds each of baseline and modern on the saved Day 4 token files.


In [ ]:
import subprocess
import sys
from pathlib import Path

import torch

INPUT = Path("/kaggle/input")
CODE = next(path.parent.parent for path in INPUT.rglob("day5.py") if path.parent.name == "octlm")
TRAIN = next(path for path in INPUT.rglob("train.bin") if path.parent.name == "data")
DATA = TRAIN.parent
TOKENIZER = DATA.parent / "bpe.json"
OUT = Path("/kaggle/working/day5/seed-spread.jsonl")
OUT.parent.mkdir(parents=True, exist_ok=True)
if not TOKENIZER.is_file():
    raise FileNotFoundError(TOKENIZER)
if not torch.cuda.is_available() or torch.cuda.get_device_capability(0) != (7, 5):
    raise RuntimeError("Select a Kaggle T4 GPU")
print("PyTorch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0), flush=True)
print("Train tokens:", TRAIN.stat().st_size // 2, flush=True)

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "octlm.day5",
        "spread",
        "--config",
        str(CODE / "configs/day5.toml"),
        "--data",
        str(DATA),
        "--tokenizer",
        str(TOKENIZER),
        "--out",
        str(OUT),
        "--device",
        "cuda:0",
    ],
    cwd=CODE,
    check=True,
)
print(OUT.read_text(), flush=True)